In [11]:
import sqlite3
import json
import os

print(os.path.exists('fermentedfoods.db'))

conn = sqlite3.connect('fermentedfoods.db')
cursor = conn.cursor()

with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = json.load(f)

True


In [ ]:
food_relations = []

def process_food_name(name):
    name.replace('_', '')
    name_words = name.split(' ')
    new_name = []
    for word in name_words:
        if not any([w.isdigit() for w in word]):
            new_name.append(word)
    return ' '.join(new_name)

for ent1type, ent1name, reltype, ent2type, ent2name, value in saved_relations:
    if not value:
        continue
    if not ent1type == 'MICROBE':
        continue

    cursor.execute(rf'SELECT sample, substrate FROM fermentedfoods WHERE LOWER(species) LIKE "{ent1name.lower()}%"')
    for sample_substrate in list(cursor.fetchall()):
        if not sample_substrate[0]:
            continue

        sample = process_food_name(sample_substrate[0])
        if not sample:
            continue
        
        substrate = process_food_name(sample_substrate[1])
        if not substrate or substrate == 'nan':
            continue

        sample_relation = ('MICROBE', ent1name, 'FOUND_IN', 'FOOD', sample)
        if not sample_relation in food_relations:
            food_relations.append(sample_relation)

        substrate_relation = ('MICROBE', ent1name, 'ACTS_ON', 'SUBSTRATE', substrate)
        if not substrate_relation in food_relations:
            food_relations.append(substrate_relation)

conn.commit()
conn.close()

print(len(food_relations))
print(food_relations[:10])

67
[('MICROBE', 'Gluconobacter oxydans', 'FOUND_IN', 'FOOD', 'water_kefir'), ('MICROBE', 'Gluconobacter oxydans', 'ACTS_ON', 'SUBSTRATE', 'sugar'), ('MICROBE', 'Gluconobacter oxydans', 'FOUND_IN', 'FOOD', 'water_kefir_figs_fresh_liquid'), ('MICROBE', 'Lactococcus lactis', 'FOUND_IN', 'FOOD', 'milk_kefir'), ('MICROBE', 'Lactococcus lactis', 'ACTS_ON', 'SUBSTRATE', 'dairy'), ('MICROBE', 'Lactococcus lactis', 'FOUND_IN', 'FOOD', 'Red Sauerkraut'), ('MICROBE', 'Lactococcus lactis', 'ACTS_ON', 'SUBSTRATE', 'vegetable'), ('MICROBE', 'Lactiplantibacillus plantarum', 'FOUND_IN', 'FOOD', 'novel miso'), ('MICROBE', 'Lactiplantibacillus plantarum', 'ACTS_ON', 'SUBSTRATE', 'legumes'), ('MICROBE', 'Latilactobacillus curvatus', 'FOUND_IN', 'FOOD', 'fermented seafoods')]


In [13]:
from neo4j import GraphDatabase

with open('..\\dissertation DLC content\\go away\\passwords.json', 'r') as f:
    passwords = dict(json.load(f))

neo4j_pw = passwords['neo4j']

del passwords

numbers_map = {0: 'zero', 1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}

def preprocess_label(label):
    if label[0].isnumeric():
        label = label.replace(label[0], numbers_map[int(label[0])])
    replace_chars = ["-", "(", ")", " ", ".", ",", "\'", "/", "\"", ";", "&", ":", ";", "|", "%", "’", "?", "!", "«", "+", "（", "）", "？", "，", "<", ">"]
    for char in replace_chars:
        label = label.replace(char, '')
    return label

def preprocess_name(name):
    replace_chars = ['[', ']', "'", '(', ')', '\"', '{', '}']
    for char in replace_chars:
        name = name.replace(char, '')
    return name

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", neo4j_pw)

driver = GraphDatabase.driver(URI, auth=AUTH, max_connection_lifetime=3600)

with driver.session() as session:

    for ent1type, ent1, rel, ent2type, ent2 in food_relations:

        ent1name = preprocess_name(ent1)
        ent1label = preprocess_label(ent1name)

        ent2name = preprocess_name(ent2)
        ent2label = preprocess_label(ent2name)

        ent1string = f"MERGE ({ent1label}:{ent1type}" + r"{" + f"customId: '{ent1label}'" f", name: '{ent1name}" + r"'})"

        session.run(ent1string)


        ent2string = f"MERGE ({ent2label}:{ent2type}" + r"{" + f"customId: '{ent2label}'" + f", name: '{ent2name}" + r"'})"

        session.run(ent2string)


        relstringneo =  f"""MATCH (a) WHERE a.customId = '{ent1label}'
                            MATCH (b) WHERE b.customId = '{ent2label}'
                            MERGE (a)-[:{rel}]->(b)
                            """
        
        session.run(relstringneo)